# OpenPlaque v0.3 — Boundary Refinement Parameter Tuning

This notebook extends the v0.2 end-to-end workflow with **parameter tuning** for plaque boundary refinement.

It assumes you have already run segmentation and have `lad_report`, `rca_report`, and `lcx_report` in memory. If not, run the v0.2 end-to-end notebook first through the segmentation cells.

Outputs saved to Google Drive:

```text
/content/drive/MyDrive/OpenPlaque/Segmentations/boundary_tuning_results.csv
/content/drive/MyDrive/OpenPlaque/Segmentations/boundary_tuning_results.json
/content/drive/MyDrive/OpenPlaque/Segmentations/boundary_tuning_report.html
/content/drive/MyDrive/OpenPlaque/Segmentations/openplaque_tuned_tpv_report.html
```

Research use only. Not for clinical decision-making.

In [ ]:
# Ensure local OpenPlaque modules are importable
import sys, os
from pathlib import Path

repo = Path('/content/OpenPlaque')
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

os.environ.setdefault('nnUNet_raw', '/content/nnUNet_raw')
os.environ.setdefault('nnUNet_preprocessed', '/content/nnUNet_preprocessed')
os.environ.setdefault('nnUNet_results', '/content/nnUNet_results')

print('Python path ready')

## 1. Confirm segmentation reports exist

This tuning notebook works on the segmentation results already produced by the end-to-end workflow.

In [ ]:
reports = [lad_report, rca_report, lcx_report]

for r in reports:
    print(r.name, 'raw TPV:', f'{r.tpv_mm3:.2f} mm³', 'voxels:', r.plaque_voxels)

## 2. Run boundary-refinement parameter tuning

The default grid tests:

- small-component removal thresholds
- lumen-adjacent trimming distances
- optional HU thresholds
- optional erosion/core settings

The objective is unsupervised. It favors conservative boundary cleanup while avoiding settings that delete too much plaque or do nothing.

In [ ]:
from openplaque.tuning import (
    tune_boundary_parameters,
    apply_selected_refinement,
    write_tuning_html_report,
)

tuning = tune_boundary_parameters(reports)

print('Selected global parameters:')
for k, v in tuning.selected_params.items():
    print(f'  {k}: {v}')

print('
Best by vessel:')
for vessel, row in tuning.best_by_vessel.items():
    print(f'{vessel}: candidate {row.candidate_id}, score={row.score:.2f}, refined={row.refined_tpv_mm3:.2f} mm³, retained={100*row.retained_fraction:.1f}%')

## 3. Save tuning tables

In [ ]:
save_dir = Path('/content/drive/MyDrive/OpenPlaque/Segmentations')
save_dir.mkdir(parents=True, exist_ok=True)

csv_path = tuning.save_csv(save_dir / 'boundary_tuning_results.csv')
json_path = tuning.save_json(save_dir / 'boundary_tuning_results.json')
html_path = write_tuning_html_report(tuning, save_dir / 'boundary_tuning_report.html')

print('Saved:', csv_path)
print('Saved:', json_path)
print('Saved:', html_path)

## 4. Inspect top candidates in a table

In [ ]:
import pandas as pd

df = pd.DataFrame(tuning.to_rows())
cols = [
    'vessel', 'candidate_id', 'score', 'min_component_voxels',
    'lumen_distance_voxels', 'high_hu_threshold', 'low_hu_threshold',
    'erode_core', 'raw_tpv_mm3', 'refined_tpv_mm3',
    'retained_fraction', 'components_before', 'components_after',
    'largest_component_fraction', 'mean_hu', 'reject_reason'
]

for vessel in df['vessel'].unique():
    print('
' + '='*80)
    print(vessel)
    display(df[df['vessel'] == vessel].sort_values('score', ascending=False)[cols].head(12))

## 5. Plot tuning sensitivity

These plots help decide whether the selected parameters are stable or whether the result is highly sensitive to one setting.

In [ ]:
import matplotlib.pyplot as plt

for vessel in df['vessel'].unique():
    sub = df[df['vessel'] == vessel].copy().sort_values('score', ascending=False).head(40)
    plt.figure(figsize=(9, 4))
    plt.plot(range(len(sub)), sub['refined_tpv_mm3'].values, marker='o')
    plt.title(f'{vessel}: refined TPV among top 40 candidates')
    plt.xlabel('Candidate rank by score')
    plt.ylabel('Refined TPV mm³')
    plt.grid(True, alpha=0.3)
    plt.show()

## 6. Apply selected tuned refinement and regenerate the TPV HTML report

In [ ]:
from openplaque.boundary import refine_plaque_mask
from openplaque.uncertainty import make_tpv_uncertainty_summary
from openplaque.report import write_html_report

# Main tuned refined masks
tuned_refined = apply_selected_refinement(reports, tuning.selected_params)

# High-confidence core remains intentionally conservative for interval lower bound
core_results = {}
for report in reports:
    core_results[report.name] = refine_plaque_mask(
        volume=report.volume,
        mask=report.mask,
        spacing=report.mask_image.GetSpacing(),
        remove_small=True,
        min_component_voxels=max(10, tuning.selected_params.get('min_component_voxels', 10)),
        trim_lumen_adjacent=True,
        lumen_distance_voxels=max(1, tuning.selected_params.get('lumen_distance_voxels', 1)),
        erode_core=True,
        erosion_iterations=1,
        high_hu_threshold=tuning.selected_params.get('high_hu_threshold'),
        low_hu_threshold=tuning.selected_params.get('low_hu_threshold'),
    )

uncertainty = make_tpv_uncertainty_summary(reports, tuned_refined, core_results)

for row in uncertainty.rows():
    print(row)

report_path = write_html_report(
    save_dir / 'openplaque_tuned_tpv_report.html',
    reports,
    tuned_refined,
    core_results,
    uncertainty,
    title='OpenPlaque Tuned TPV Boundary Refinement Report',
)
print('Saved:', report_path)

## 7. Optional visual QC of tuned masks

In [ ]:
for report in reports:
    print('
' + '='*80)
    print(report.name)
    tuned_refined[report.name].summary()
    tuned_refined[report.name].show_refined_overlay(report.volume)
    tuned_refined[report.name].show_removed_overlay(report.volume)